<a href="https://colab.research.google.com/github/ParkHangah/AIFFEL_quest_eng/blob/master/Model_Serving/MS08/Global_Narrative_to_Tag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Global Narrative-to-Tag

코랩 런타임 버전:  2025.07 [상세보기](https://research.google.com/colaboratory/runtime-version-faq.html#2026.01)

### [프로젝트 요구사항]

#### 1.1 필수 구현 항목:

**FastAPI 백엔드**
   - 추론 엔드포인트 (POST /predict)
   - Pydantic으로 입력 검증
   - 비동기 추론 (run_in_executor)   

**API Key 인증**
   - Day 6의 auth.py 재사용  

**Streamlit 프론트엔드**
   - 사용자 입력 → API 호출 → 결과 표시  
   
**에러 처리**
   - 잘못된 입력, 모델 에러 시 적절한 HTTP 상태 코드 반환

#### 1.2 제한 사항

- 모델 학습 (사전학습 모델을 가져다 씁니다)
- Docker 패키징 (MLOps 과정에서 다룹니다)
- 데이터베이스 연동

#### 1.3 평가 기준

✅ 서버가 정상적으로 실행되는가?  
✅ Swagger UI에서 추론이 동작하는가?  
✅ API Key 없이 요청하면 401이 반환되는가?  
✅ 잘못된 입력에 대해 적절한 에러 메시지가 나오는가?  
✅ Streamlit UI에서 입력 → 결과 확인이 가능한가?  

## 0.🚀 프로젝트 설계

### 0-0. 프로젝트 정의서: Global Narrative-to-Tag (PoC)

#### **1) 프로젝트 목표**
최대 5000자의 웹소설/웹툰 대본(서사 텍스트)을 입력받아, 단락별로 다국어 번역과 핵심 키워드(태그) 추출을 동시에 수행하는 지능형 현지화 파이프라인의 프로토타입 구축

#### **2) 핵심 구현 기능 (기능 명세)**
*   **프론트엔드 (Streamlit):**
    *   최대 5000자를 입력받을 수 있는 넓은 텍스트 박스(`st.text_area`) 제공.
    *   원문 작품을 [ [단락코드,[문장1,문장2, ... , 문장N]], [단락코드',[문장1',문장2',..., 문장N'], ...] 형태의 리스트로 변환 (시간 절약을 위해 복잡한 NLP 문장 분리기 대신, 파이썬 기본 줄바꿈(\n}이나 마치미표(.)을 기준으로 단란을 쪼개기)
    *  사전 학습 모델을 사용 문장 분석 하여 [[단락코드,[원본 문장, 번역된 문장,[추출된 태그]]] 형태의 리스트로 결과 출력.
*   **백엔드 (FastAPI + run_in_executor):**
    *   `POST /analyze` 엔드포인트 구현.
    *   5000자의 긴 텍스트를 처리하는 동안 서버가 멈추지 않도록(Blocking 방지) 추론 작업을 `ThreadPoolExecutor`와 `run_in_executor`를 활용해 별도 스레드에서 처리.
*   **AI 파이프라인 (Hugging Face):**
    *   **Translation:** 한국어 단락을 영어 등 타겟 언어로 번역.
    *   **Zero-Shot Tagging:** 각 단락을 미리 정의해 둔 후보 키워드(예: `["액션", "로맨스", "심리", "반전", "배경묘사"]`) 중 확률이 가장 높은 것으로 매핑.

### 0-1. 백엔드 설계

#### 1) 텍스트 입력 및 전처리
- 사용자가 텍스트를 입력하거나 txt 파일을 불러오면 \n과 .을 기준으로 문단별 문장으로 분리.
- [[[문장타입, 문장1], [문장타입, 문장2], ...], ...] 형태의 리스트로 변경.
- 문장 타입 분류: 일반(0), 대사(" 쌍따옴표로 감싸진 문장, 1), 생각(' 작은따옴표로 감싸진 문장, 2).
- 불필요한 특수 문자 모두 제거.
- 파일 저장: data 폴더 안에 VALID_API_KEYS의 사용자명으로 새로운 폴더를 생성하여 파일 저장.
- 파일명 규칙: YYYYMMDDHHMMSS + 랜덤 3자리 숫자.txt 형식으로 지정하여 같은 폴더에 저장.
#### 2) 문장 추출
- 1단계에서 만들어진 문단별 문장 리스트에서 문장만 별도의 리스트([])로 추출.
#### 3) 번역 변환 (translator_model)
- 번역된 결과를 translator_result = [[문장타입, [원문문장, 번역문장]], ...] 형태로 변환.
#### 4) 키워드 추출 (quantized_model)
- 원문 문장과 번역 문장의 키워드를 추출. (문장별로 추출 시 개수가 너무 많아지므로 문단별 추출 진행)
- quantized_result = [[[[문장타입, 원문문장, 번역문장], [문장타입, 원문문장, 번역문장]], [원문키워드, 번역문장키워드]], ...] 형태로 저장.
#### 5) 결과 딕셔너리 생성
- 전체 문단 및 문장에 대한 키워드 중 빈도수가 가장 높은 원문 키워드와 번역 문장 키워드를 상위 5개씩 추출.
- 결과 포맷: {'title': "작품제목", 'origin_txt': "data의 user 폴더 안에 생성된 txt 파일 path", 'keyword': [[원문키워드], [영문키워드]], 'data': quantized_result}
#### 6) JSON 파일 저장
- 5단계에서 생성한 딕셔너리를 JSON 형태로 변환.
- data 폴더 내 해당 사용자(user)의 폴더에 저장하되, 파일명은 오리지널 txt 파일 이름과 동일하게 설정.


### 0-2. 프론트엔드 설계

#### 1) 우측 사이드바
- API 키 입력창
- 내가 그동안 작성한 작품 리스트 선택 메뉴
#### 2) 메인 화면 (입력부)
- 서비스 타이틀 및 서비스 소개
- 작품 제목 입력창
- 기능 탭 구성:
  - [파일로 불러오기] 탭: 파일 선택창, 확인 버튼
  - [작품 입력하기] 탭: 텍스트 입력창 (크기 조절 가능 + 글자 수 세기 지원), 임시저장 버튼 (임시저장 시 status에 저장), 임시저장된 파일 불러오기 버튼, 확인 버튼
#### 3) 작품 출력 화면 (결과부)
- 작품 제목
- 출력 내용 선택 버튼: 원문 보기, 번역본 보기, 원문+번역 대조 보기
- 출력 형태 선택 버튼: <p> 태그를 원래 display 형태인 block으로 보여줄지, inline으로 보여줄지 선택
- 작품 키워드: 출력 형태 선택에 따라 원문 키워드, 번역본 키워드, 혹은 둘 다(원문+번역) 출력
#### 4) 작품 렌더링 규칙
- 모든 문단은 `<section>` 태그로 감싸기.
- 각 문장은 `<p>` 태그로 감싸기.
- 대사나 생각 문장에는 class 속성을 추가하고, 해당 속성에 따라 ::before와 ::after 가상 요소를 활용해 작은따옴표(')와 쌍따옴표(")를 화면에 출력.
- 대사나 생각 문장은 무조건 display: block으로 설정.




## 01. 사전 준비

### 📍프로젝트 경로 설정

In [3]:
drive_path = 'Colab Notebooks/Aiffel' # 평상시 작업하는 드라이브 폴더 경로를 입력해 주세요.
project_name = 'Global Narrative-to-Tag'       # 이번 프로젝트세 사용하는 폴더명을 입력해주세요.

In [4]:
from google.colab import drive
from IPython.display import clear_output
import os
# 1. 구글 드라이브 마운트
print("[0;33mConnecting...")
drive.mount('/content/gdrive')
# 2. 경로 설정 및 폴더 생성
project_path = os.path.join('/content/gdrive/MyDrive',drive_path, project_name)
# Create the project directory if it doesn't exist
os.makedirs(project_path, exist_ok=True)
clear_output()
print(f"✅ Google Drive 연결 성공!! ( project_path: {project_path} )")

✅ Google Drive 연결 성공!! ( project_path: /content/gdrive/MyDrive/Colab Notebooks/Aiffel/Global Narrative-to-Tag )


In [5]:
%cd "{project_path}"

/content/gdrive/MyDrive/Colab Notebooks/Aiffel/Global Narrative-to-Tag


### 📃requirements.txt 파일 만들기

#### old

In [5]:
# %%writefile requirements.txt
# # ===== Core =====
# torch>=2.1.0,<2.5.0
# torchvision>=0.16.0,<0.20.0

# # ===== LLM & Quantization =====
# transformers
# accelerate
# bitsandbytes>=0.39.0

# # ===== Translation & Tokenization (새로 추가된 부분) =====
# sentencepiece      # 다국어 번역 모델 토크나이저에 필수
# protobuf           # sentencepiece 및 일부 모델 동작에 필수

# # ===== API =====
# fastapi==0.115.0
# uvicorn[standard]==0.30.0
# pydantic>=2.0.0,<3.0.0

# # ===== Frontend =====
# streamlit==1.38.0

# # ===== Utilities =====
# nest-asyncio
# requests>=2.31.0,<3.0.0
# pillow>=10.0.0
# python-multipart>=0.0.6

Writing requirements.txt


#### 의존성 문제 해결을 위한 New 버전

In [5]:
%%writefile requirements.txt
# ===== Core =====
# 1. PyTorch 보안 이슈(CVE-2025-32434) 해결을 위해 2.6.0 이상 강제
torch>=2.6.0
torchvision>=0.16.0

# ===== LLM & Quantization =====
# 2. bitsandbytes 0.44.1과 완벽히 호환되며 번역 파이프라인이 안정적인 버전으로 고정
transformers==4.44.2
accelerate
# 3. Colab CUDA 12.x 환경과의 충돌(libnvJitLink.so.13 에러) 방지
bitsandbytes==0.44.1

# ===== Translation & Tokenization =====
sentencepiece
protobuf
sacremoses         # 4. Helsinki-NLP 번역 파이프라인(MarianMT) 필수 의존성 추가

# ===== API =====
fastapi==0.115.0
uvicorn[standard]==0.30.0
pydantic>=2.0.0,<3.0.0

# ===== Frontend =====
streamlit==1.38.0

# ===== Utilities =====
nest-asyncio
requests>=2.31.0,<3.0.0
pillow>=10.0.0
python-multipart>=0.0.6

Overwriting requirements.txt


In [6]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 117.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.4/122.4 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.6/94.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 127.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 171.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 138.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## 02. 모델 선택

### 다국어 기계 번역 (한국어 → 영어)

- 모델명: Helsinki-NLP/opus-mt-ko-en
- Model page: https://huggingface.co/Helsinki-NLP/opus-mt-ko-en
- opus_readme_url: https://github.com/Helsinki-NLP/Tatoeba-Challenge/tree/master/models/kor-eng/README.md
- repo_id: Helsinki-NLP/opus-mt-ko-en

#### try 1 (최종: 의존성 문제 해결을 위해 라이브러리 버전 낮춤)

In [6]:
from transformers import pipeline
# 본인이 선택한 모델로 교체하세요
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-ko-en")
# 2. 번역 테스트
sample_text = "주인공은 어두운 숲 속에서 전설의 검을 발견했다."
translated_result =translator(sample_text)
print(translated_result)
# [{'label': 'positive', 'score': 0.95}]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


[{'translation_text': 'The hero found the sword of legend in the dark forest.'}]


#### try 2: transformers 버전 낮춤

```
# Use a pipeline as a high-level helper
# Warning: Pipeline type "translation" is no longer supported in transformers v5.
# You must load the model directly (see below) or downgrade to v4.x with:
# 'pip install "transformers<5.0.0'
```

In [ ]:
# from transformers import pipeline
# # 본인이 선택한 모델로 교체하세요
# translator = pipeline("translation", model="Helsinki-NLP/opus-mt-ko-en")
# # 2. 번역 테스트
# sample_text = "주인공은 어두운 숲 속에서 전설의 검을 발견했다."
# translated_result =translator(sample_text)
# print(translated_result)
# # [{'label': 'positive', 'score': 0.95}]

#### try3: Seq2Seq 활용

In [4]:
# from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
# import torch

# model_id = "Helsinki-NLP/opus-mt-ko-en"
# device = "cuda" if torch.cuda.is_available() else "cpu"

# # 1. 토크나이저와 번역 모델 직접 로드
# # (번역 모델은 Seq2Seq 구조이므로 AutoModelForSeq2SeqLM을 사용합니다)
# translator_tokenizer = AutoTokenizer.from_pretrained(model_id)
# translator_model = AutoModelForSeq2SeqLM.from_pretrained(
#     model_id,
#     use_safetensors=True
# ).to(device)
# # 2. 번역 테스트
# sample_text = ["주인공은 어두운 숲 속에서 전설의 검을 발견했다.", "그리고 마왕을 향해 걸어갔다."]
# # 텍스트를 토큰화하여 디바이스(GPU)로 이동 (패딩 추가)
# inputs = translator_tokenizer(sample_text, return_tensors="pt", padding=True, truncation=True).to(device)

# # 모델 추론 (generate)
# with torch.no_grad():
#     outputs = translator_model.generate(**inputs)

# # 2차원 배열(여러 문장)을 그대로 처리하기 위해 batch_decode 사용
# translated_texts = translator_tokenizer.batch_decode(outputs, skip_special_tokens=True)

# print(translated_texts)
# # ['The protagonist found a sword in the forest.', 'And walked towards the demon king.']

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/842k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


OSError: Can't load the model for 'Helsinki-NLP/opus-mt-ko-en'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'Helsinki-NLP/opus-mt-ko-en' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.

### 제로샷 카테고리 분류 (다국어 지원)

- 모델명: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
- Model page: https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-mnli-xnli

##### ☢️ 파이토치 업데이트로 인한 충돌 해결을 위한 라이브러리 다운그레이 (requirements.txt 수정으로 인해 사용 안함):

In [6]:
# !pip uninstall -y transformers bitsandbytes
# !pip install transformers==4.47.1 bitsandbytes==0.44.1 accelerate

Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: bitsandbytes 0.44.1
Uninstalling bitsandbytes-0.44.1:
  Successfully uninstalled bitsandbytes-0.44.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.3 MB/s eta 0:00:00
  Using cached bitsandbytes-0.44.1-py3-none-manylinux_2_24_x86_64.whl.metadata (3.5 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 101.9 MB/s eta 0:00:00
Using cached bitsandbytes-0.44.1-py3-none-manylinux_2_24_x86_64.whl (122.4 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 131.7 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.9.0
    Uninstalling huggingface_hub-1.9.0:
      Successfully uninstalled huggingface_hub-1.9.0
  Attempting uni

##### 📟 기본 코드

In [12]:
import torch
from transformers import pipeline


model_id = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"


# DeBERTa 모델은 device_map="auto"를 지원하지 않으므로 GPU 사용을 위해 명시적으로 device 인자를 지정합니다.
device_id = 0 if torch.cuda.is_available() else -1


# 2. 파이프라인에 직접 모델 로드
classifier = pipeline(
    "zero-shot-classification",
    model=model_id,
    device=device_id
)


# 3. 오프라인 LLM으로 미리 뽑아둔 '후보 키워드' (예시)
candidate_labels = ["액션", "마법", "성장", "반전", "배경묘사"]


# 4. 분류 테스트
# (sample_text는 이전 셀에서 정의된 변수를 사용합니다)
if 'sample_text' not in locals():
    sample_text = "주인공은 어두운 숲 속에서 전설의 검을 발견했다."


results = classifier(sample_text, candidate_labels)


print("\n=== 제로샷 분류(태깅) 결과 ===")
# 결과가 단일 딕셔너리인지 리스트인지 확인 후 반복문 처리
if isinstance(results, dict):
    results = [results]


for result in results:
    print(f"원문: {result['sequence']}")
    print(f"가장 높은 확률의 태그: {result['labels'][0]} (점수: {result['scores'][0]:.4f})")
    print(f"두 번째 높은 확률의 태그: {result['labels'][1]} (점수: {result['scores'][1]:.4f})\n")


=== 제로샷 분류(태깅) 결과 ===
원문: 주인공은 어두운 숲 속에서 전설의 검을 발견했다.
가장 높은 확률의 태그: 마법 (점수: 0.7627)
두 번째 높은 확률의 태그: 배경묘사 (점수: 0.0921)



##### 📟 양자화 적용

In [11]:
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer

model_id = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

# 1. 4비트 양자화로 모델 로드 (가장 중요한 부분) [1]
# load_in_4bit=True로 메모리를 절약하고, device_map="auto"로 GPU에 자동 할당합니다 [1].
quantized_model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    load_in_4bit=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. 로드된 양자화 모델을 pipeline에 주입
classifier = pipeline(
    "zero-shot-classification",
    model=quantized_model,
    tokenizer=tokenizer
)

# 3. 오프라인 LLM으로 미리 뽑아둔 '후보 키워드' (예시)
candidate_labels = ["액션", "마법", "성장", "반전", "배경묘사"]

# 4. 분류 테스트
result = classifier(sample_text, candidate_labels)

print("\n=== 제로샷 분류(태깅) 결과 ===")
print(f"원문: {result['sequence']}")
print(f"가장 높은 확률의 태그: {result['labels']} (점수: {result['scores']:.4f})")
print(f"두 번째 높은 확률의 태그: {result['labels'][2]} (점수: {result['scores'][2]:.4f})")

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


RuntimeError: Failed to import transformers.integrations.bitsandbytes because of the following error (look up to see its traceback):
No module named 'triton.ops'

##### 📟 양자화 적용 import 에러 우회 :  파이썬 코드 최상단에서 가짜 모듈(Monkey Patch)을 만들어 속이는 방법

In [17]:
# =================================================================
# 1. Triton Import 에러 완벽 우회를 위한 몽키 패치 (MagicMock 사용)
# =================================================================
import sys
from unittest.mock import MagicMock

try:
    import triton
    import triton.language
except ImportError:
    pass

# bitsandbytes가 요구하는 과거 버전의 triton.ops 모듈과 내부 함수들을 모두 MagicMock으로 대체
mock_ops = MagicMock()
sys.modules['triton.ops'] = mock_ops
sys.modules['triton.ops.matmul_perf_model'] = mock_ops

# =================================================================
# 2. 라이브러리 로드 및 모델 셋업
# =================================================================
import torch
from transformers import pipeline, AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig

# 4비트 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"

print("양자화 모델 로드 중...")
# 양자화 설정 적용
quantized_model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 파이프라인 생성
classifier = pipeline(
    "zero-shot-classification",
    model=quantized_model,
    tokenizer=tokenizer
)

# =================================================================
# 3. 추론 테스트
# =================================================================
candidate_labels = ["액션", "마법", "성장", "반전", "배경묘사"]
sample_text = "주인공은 어두운 숲 속에서 전설의 검을 발견했다."

print("추론 시작...")
result = classifier(sample_text, candidate_labels)

print("\n=== 제로샷 분류(태깅) 결과 ===")
print(f"원문: {result['sequence']}")
print(f"가장 높은 확률의 태그: {result['labels'][0]} (점수: {result['scores'][0]:.4f})")
print(f"두 번째 높은 확률의 태그: {result['labels'][1]} (점수: {result['scores'][1]:.4f})")

양자화 모델 로드 중...


ValueError: DebertaV2ForSequenceClassification does not support `device_map='auto'`. To implement support, the model class needs to implement the `_no_split_modules` attribute.

## 3. 프로젝트 뼈대

### 3-1.📁프로젝트 폴더 구성 세팅

In [15]:
import os

folders = [
    "app",             # FastAPI 애플리케이션 코드
    "frontend",        # Streamlit 프론트엔드 코드
    "models",
    "data",            # 샘플 데이터
    "tests",           # 테스트 코드
]
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✅ {folder}/ 생성 완료")

✅ app/ 생성 완료
✅ frontend/ 생성 완료
✅ models/ 생성 완료
✅ data/ 생성 완료
✅ tests/ 생성 완료


### 3-2.📜auth.py — 재사용

In [24]:
%%writefile app/auth.py
"""
Day 6에서 만든 인증 모듈을 그대로 재사용합니다.
"""
from fastapi import HTTPException, Header

VALID_API_KEYS = {
    "test-key-001": "user01",
    "test-key-002": "user02",
}


async def verify_api_key(x_api_key: str = Header(None)) -> str:
    if x_api_key is None:
        raise HTTPException(
            status_code=401,
            detail="API Key가 필요합니다. X-API-Key 헤더를 포함해 주세요.",
        )
    if x_api_key not in VALID_API_KEYS:
        raise HTTPException(
            status_code=401,
            detail="유효하지 않은 API Key입니다.",
        )
    return VALID_API_KEYS[x_api_key]

Writing app/auth.py


### 3-3.📝schemas.py — 직접 작성

In [26]:
%%writefile app/schemas.py
"""
입력/출력 스키마.

"""
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

# ---------------------------------------------------------
# [요청 스키마]
# ---------------------------------------------------------
class NarrativeRequest(BaseModel):
    """
    사용자로부터 텍스트를 입력받는 요청 모델 (단계 1의 입력)
    """
    title: str = Field(..., description="작품 제목 (필수)")
    # min_length, max_length로 길이를 엄격히 검증
    text: str = Field(..., min_length=1, max_length=5000, description="분석할 서사 텍스트 (최대 5000자) (필수)")


# ---------------------------------------------------------
# [응답 데이터 세부 구조]
# ---------------------------------------------------------
class SentenceDetail(BaseModel):
    """
    개별 문장 분석 결과 구조
    """
    # ge(greater or equal), le(less or equal)를 사용하여 범위 검증 추가
    sentence_type: int = Field(..., ge=0, le=2, description="0: 일반, 1: 대사, 2: 생각 (필수)")
    original_text: str = Field(..., description="원문 문장 (필수)")
    # 번역에 실패하거나 번역이 제공되지 않을 경우를 대비한 선택 필드 처리
    translated_text: Optional[str] = Field(default=None, description="번역된 문장 (선택)")


class ParagraphDetail(BaseModel):
    """
    문단별 분석 결과 구조 (단계 4의 quantized_result 단위)
    """
    sentences: List[SentenceDetail] = Field(..., description="문단 내 문장들 (필수)")
    # 모델 추론 결과에 따라 키워드가 없을 수도 있으므로 선택 필드 처리
    keywords: Optional[List[List[str]]] = Field(default=None, description="[원문키워드 리스트, 영문키워드 리스트] (선택)")


# ---------------------------------------------------------
# [최종 응답 스키마]
# ---------------------------------------------------------
class NarrativeResponse(BaseModel):
    """
    API의 최종 응답 모델 (단계 5, 6의 딕셔너리 구조)
    """
    title: str = Field(..., description="작품 제목 (필수)")
    # 파일 저장을 선택적 기능으로 처리할 경우를 가정한 선택 필드 처리
    origin_txt: Optional[str] = Field(default=None, description="data 폴더 안에 생성된 txt 파일 path (선택)")
    keyword: Optional[List[List[str]]] = Field(default=None, description="전체 상위 5개 추출 키워드 [[원문키워드], [영문키워드]] (선택)")
    data: List[ParagraphDetail] = Field(..., description="문단별 상세 분석 데이터 (quantized_result) (필수)")

Overwriting app/schemas.py


### 3-4.📝model_service.py — 직접 작성

In [7]:
# keyword.txt 파일을 읽어 리스트에 담는 코드
with open('data/keyword.txt', 'r', encoding='utf-8') as file:
    content = file.read()
    # 쉼표(,)로 구분하고 각 키워드의 양쪽 공백을 제거하여 리스트로 생성
    keyword_list = [keyword.strip() for keyword in content.split(',')]

print(keyword_list)

['로맨스', '현대', '정통', '판타지', '무협', '미스터리', '스릴러', '아포칼립스', '일상', '코미디', '복수', '성장', '힐링', '피폐', '코믹', '잔잔', '계약연애', '정략결혼', '첫사랑', '쌍방구원', '오해물', '역하렘', '하렘', '신분차이', '능력여주', '회귀', '빙의', '다정', '츤데레', '냉미남', '냉미녀', '집착', '광공', '후회', '순정', '직진', '능글', '상처', '흑막', '천재', '사이다전개', '두뇌싸움', '심리전', '반전', '세계관탄탄', '떡밥회수', '감정선탄탄', '다크판타지', '느와르', '동화', '신화', '마법', '요정', 'SF', '권성징악', '비극', '희극', '서정적', '철학적', '사회비판', '우정', '사랑', '운명', '옴니버스', '영웅', '동양풍', '역사', '대체역사', '동화', '수인', '인외존재']


#### ( try 1 )

In [39]:
# %%writefile app/model_service.py
# """
# 모델 로드와 추론 함수
# """
# import os
# import re
# import json
# import random
# from datetime import datetime
# from collections import Counter
# from transformers import pipeline, BitsAndBytesConfig

# # 스크립트 내부에서 직접 keyword.txt 파일을 읽어 리스트에 담습니다.
# try:
#     with open('data/keyword.txt', 'r', encoding='utf-8') as file:
#         content = file.read()
#         KOREAN_KEYWORDS = [keyword.strip() for keyword in content.split(',')]
# except FileNotFoundError:
#     # 파일이 없을 경우를 대비한 기본 키워드
#     KOREAN_KEYWORDS = ["로맨스", "판타지", "액션", "무협", "스릴러"]

# # (Zero-Shot 분류를 위한 영문 키워드 매핑 - 필요시 확장)
# ENGLISH_KEYWORDS = ["Romance Fantasy", "Modern Romance", "Traditional Fantasy", "Modern Fantasy", "Martial Arts", "Mystery", "Thriller", "Apocalypse", "Slice of Life", "RomCom", "Revenge", "Coming-of-age", "Healing", "Angst", "Comedy", "Calm", "Contract Dating", "Arranged Marriage", "First Love", "Mutual Salvation", "Misunderstanding", "Reverse Harem", "Harem", "Status Difference", "Capable FL", "Girl Crush FL", "Regression FL", "Reincarnation FL", "Possession FL", "Sweet ML", "Tsundere ML", "Cold Handsome ML", "Obsessive ML", "Crazy Obsessive ML", "Regretful ML", "Pure Love ML", "Straightforward ML", "Sly ML", "Scarred FL", "Mastermind ML", "Genius ML", "Satisfying Plot", "Brain Battle", "Psychological", "Plot Twist", "Solid World-building", "Foreshadowing Retrieval", "Solid Emotional Line", "Dark Fantasy", "Noir"]

# def load_model():
#     """
#     모델을 로드하여 반환합니다.
#     - 번역 모델: 가벼운 다국어 번역 모델
#     - 제로샷 분류 모델: 4비트 양자화(OOM 방지) 로드
#     """
#     # 번역 모델 로드 (Helsinki-NLP 등) - safetensors 명시적 사용
#     translator = pipeline(
#         "translation",
#         model="Helsinki-NLP/opus-mt-ko-en",
#         device_map="auto",
#         model_kwargs={"use_safetensors": True}
#     )

#     # 제로샷 모델 4비트 로드 (BitsAndBytes)
#     quantization_config = BitsAndBytesConfig(load_in_4bit=True)
#     tagger = pipeline(
#         "zero-shot-classification",
#         model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
#         model_kwargs={"quantization_config": quantization_config},
#         device_map="auto"
#     )

#     return {"translator": translator, "tagger": tagger}

# def predict(model: dict, input_data: dict) -> dict:
#     """
#     입력을 받아 1~6 단계 로직을 수행하고 추론 결과를 반환합니다.
#     """
#     translator = model["translator"]
#     tagger = model["tagger"]

#     text = input_data["text"]
#     title = input_data["title"]
#     username = input_data["username"]

#     # --- [로직 1] 파일명 생성 및 데이터 폴더 저장 ---
#     os.makedirs(f"data/{username}", exist_ok=True)
#     base_filename = datetime.now().strftime("%Y%m%d%H%M%S") + str(random.randint(100, 999))
#     txt_path = f"data/{username}/{base_filename}.txt"

#     with open(txt_path, "w", encoding="utf-8") as f:
#         f.write(text)

#     # --- [로직 1, 2] 문단/문장 분리 및 정제 ---
#     paragraphs = text.split('\n')
#     parsed_paragraphs = []

#     for p in paragraphs:
#         if not p.strip(): continue
#         # 마침표(.) 단위로 문장 분리
#         sentences = [s.strip() + '.' for s in p.split('.') if s.strip()]
#         para_sents = []
#         for s in sentences:
#             # 불필요한 특수 문자 제거
#             clean_s = re.sub(r'[^a-zA-Z0-9가-힣\s\'".,!?]', '', s).strip()
#             if not clean_s: continue

#             # 문장 타입 판별 (0=일반, 1=대사, 2=생각)
#             s_type = 0
#             if clean_s.startswith('"') and clean_s.endswith('"'):
#                 s_type = 1
#             elif clean_s.startswith("'") and clean_s.endswith("'"):
#                 s_type = 2
#             para_sents.append([s_type, clean_s])

#         if para_sents:
#             parsed_paragraphs.append(para_sents)

#     # --- [로직 3, 4] 문단 단위 번역 & 제로샷 태깅 ---
#     quantized_result = []
#     all_orig_keywords = []
#     all_trans_keywords = []

#     for para in parsed_paragraphs:
#         para_translated = []
#         para_orig_text = ""
#         para_trans_text = ""

#         # 문단 내 문장별 번역 처리
#         for s_type, orig_sent in para:
#             trans_sent = translator(orig_sent)['translation_text']
#             para_translated.append([s_type, orig_sent, trans_sent])
#             para_orig_text += orig_sent + " "
#             para_trans_text += trans_sent + " "

#         # 문단 단위로 가장 확률 높은 키워드(상위 3개) 추출
#         orig_tags = tagger(para_orig_text, KOREAN_KEYWORDS, multi_label=True)
#         trans_tags = tagger(para_trans_text, ENGLISH_KEYWORDS, multi_label=True)

#         para_orig_keys = orig_tags['labels'][:3]
#         para_trans_keys = trans_tags['labels'][:3]

#         all_orig_keywords.extend(para_orig_keys)
#         all_trans_keywords.extend(para_trans_keys)

#         # quantized_result 구조화
#         quantized_result.append([para_translated, [para_orig_keys, para_trans_keys]])

#     # --- [로직 5] 전체 키워드 상위 5개 추출 및 딕셔너리 구성 ---
#     top_orig = [k for k, v in Counter(all_orig_keywords).most_common(5)]
#     top_trans = [k for k, v in Counter(all_trans_keywords).most_common(5)]

#     final_dict = {
#         "title": title,
#         "origin_txt": txt_path,
#         "keyword": [top_orig, top_trans],
#         "data": quantized_result
#     }

#     # --- [로직 6] JSON 형태로 동일한 폴더, 이름으로 저장 ---
#     json_path = f"data/{username}/{base_filename}.json"
#     with open(json_path, "w", encoding="utf-8") as f:
#         json.dump(final_dict, f, ensure_ascii=False, indent=2)

#     return final_dict


Overwriting app/model_service.py


#### ( try 2 )

In [18]:
%%writefile app/model_service.py
"""
모델 로드와 추론 함수
"""
import os
import re
import json
import random
from datetime import datetime
from collections import Counter
from transformers import pipeline, BitsAndBytesConfig

# 스크립트 내부에서 직접 keyword.txt 파일을 읽어 리스트에 담습니다.
try:
    with open('data/keyword.txt', 'r', encoding='utf-8') as file:
        content = file.read()
        KOREAN_KEYWORDS = [keyword.strip() for keyword in content.split(',')]
except FileNotFoundError:
    # 파일이 없을 경우를 대비한 기본 키워드
    KOREAN_KEYWORDS = ["로맨스", "판타지", "액션", "무협", "스릴러"]

# (Zero-Shot 분류를 위한 영문 키워드 매핑 - 필요시 확장)
ENGLISH_KEYWORDS = ["Romance Fantasy", "Modern Romance", "Traditional Fantasy", "Modern Fantasy", "Martial Arts", "Mystery", "Thriller", "Apocalypse", "Slice of Life", "RomCom", "Revenge", "Coming-of-age", "Healing", "Angst", "Comedy", "Calm", "Contract Dating", "Arranged Marriage", "First Love", "Mutual Salvation", "Misunderstanding", "Reverse Harem", "Harem", "Status Difference", "Capable FL", "Girl Crush FL", "Regression FL", "Reincarnation FL", "Possession FL", "Sweet ML", "Tsundere ML", "Cold Handsome ML", "Obsessive ML", "Crazy Obsessive ML", "Regretful ML", "Pure Love ML", "Straightforward ML", "Sly ML", "Scarred FL", "Mastermind ML", "Genius ML", "Satisfying Plot", "Brain Battle", "Psychological", "Plot Twist", "Solid World-building", "Foreshadowing Retrieval", "Solid Emotional Line", "Dark Fantasy", "Noir"]

def load_model():
    """
    모델을 로드하여 반환합니다.
    """
    import torch # device 설정을 위해 상단 혹은 함수 내에 추가

    # GPU 사용 가능 여부 확인 (0: GPU, -1: CPU)
    device = 0 if torch.cuda.is_available() else -1

    # 1. 번역 모델 로드 (Helsinki-NLP)
    # framework="pt"를 명시하여 PyTorch로 로드를 강제하고,
    # 모델이 작으므로 device_map 대신 device 인덱스를 사용합니다.
    translator = pipeline(
        "translation",
        model="Helsinki-NLP/opus-mt-ko-en",
        framework="pt",               # 핵심: PyTorch 프레임워크 강제 사용
        device=device,                # device_map="auto" 대신 직접 할당
        model_kwargs={"use_safetensors": True}
    )

    # 2. 제로샷 모델 4비트 로드 (BitsAndBytes)
    # Tagger는 이미 quantized_model을 넘겨받으므로 기존 로직 유지 가능
    from transformers import BitsAndBytesConfig
    quantization_config = BitsAndBytesConfig(load_in_4bit=True)

    tagger = pipeline(
        "zero-shot-classification",
        model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
        model_kwargs={"quantization_config": quantization_config},
        device_map="auto" # Tagger는 비정형 모델이므로 device_map 사용 유지 가능 (PT 강제됨)
    )

    return {"translator": translator, "tagger": tagger}

def predict(model: dict, input_data: dict) -> dict:
    """
    입력을 받아 1~6 단계 로직을 수행하고 추론 결과를 반환합니다.
    """
    translator = model["translator"]
    tagger = model["tagger"]

    text = input_data["text"]
    title = input_data["title"]
    username = input_data["username"]

    # --- [로직 1] 파일명 생성 및 데이터 폴더 저장 ---
    os.makedirs(f"data/{username}", exist_ok=True)
    base_filename = datetime.now().strftime("%Y%m%d%H%M%S") + str(random.randint(100, 999))
    txt_path = f"data/{username}/{base_filename}.txt"

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(text)

    # --- [로직 1, 2] 문단/문장 분리 및 정제 ---
    paragraphs = text.split('\n')
    parsed_paragraphs = []

    for p in paragraphs:
        if not p.strip(): continue
        # 마침표(.) 단위로 문장 분리
        sentences = [s.strip() + '.' for s in p.split('.') if s.strip()]
        para_sents = []
        for s in sentences:
            # 불필요한 특수 문자 제거
            clean_s = re.sub(r'[^a-zA-Z0-9가-힣\s\'".,!?]', '', s).strip()
            if not clean_s: continue

            # 문장 타입 판별 (0=일반, 1=대사, 2=생각)
            s_type = 0
            if clean_s.startswith('"') and clean_s.endswith('"'):
                s_type = 1
            elif clean_s.startswith("'") and clean_s.endswith("'"):
                s_type = 2
            para_sents.append([s_type, clean_s])

        if para_sents:
            parsed_paragraphs.append(para_sents)

    # --- [로직 3, 4] 문단 단위 번역 & 제로샷 태깅 ---
    quantized_result = []
    all_orig_keywords = []
    all_trans_keywords = []

    for para in parsed_paragraphs:
        para_translated = []
        para_orig_text = ""
        para_trans_text = ""

        # 문단 내 문장별 번역 처리
        for s_type, orig_sent in para:
            trans_sent = translator(orig_sent)['translation_text']
            para_translated.append([s_type, orig_sent, trans_sent])
            para_orig_text += orig_sent + " "
            para_trans_text += trans_sent + " "

        # 문단 단위로 가장 확률 높은 키워드(상위 3개) 추출
        orig_tags = tagger(para_orig_text, KOREAN_KEYWORDS, multi_label=True)
        trans_tags = tagger(para_trans_text, ENGLISH_KEYWORDS, multi_label=True)

        para_orig_keys = orig_tags['labels'][:3]
        para_trans_keys = trans_tags['labels'][:3]

        all_orig_keywords.extend(para_orig_keys)
        all_trans_keywords.extend(para_trans_keys)

        # quantized_result 구조화
        quantized_result.append([para_translated, [para_orig_keys, para_trans_keys]])

    # --- [로직 5] 전체 키워드 상위 5개 추출 및 딕셔너리 구성 ---
    top_orig = [k for k, v in Counter(all_orig_keywords).most_common(5)]
    top_trans = [k for k, v in Counter(all_trans_keywords).most_common(5)]

    final_dict = {
        "title": title,
        "origin_txt": txt_path,
        "keyword": [top_orig, top_trans],
        "data": quantized_result
    }

    # --- [로직 6] JSON 형태로 동일한 폴더, 이름으로 저장 ---
    json_path = f"data/{username}/{base_filename}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(final_dict, f, ensure_ascii=False, indent=2)

    return final_dict


Overwriting app/model_service.py


#### ( try 3 ) 🛠️ app/model_service.py 수정 (양자화 제거 및 옵션 안정화)

In [21]:
%%writefile app/model_service.py
"""
모델 로드와 추론 함수 (양자화 제거 버전)
"""
import os
import re
import json
import random
import torch
from datetime import datetime
from collections import Counter
from transformers import pipeline

try:
    with open('data/keyword.txt', 'r', encoding='utf-8') as file:
        content = file.read()
        KOREAN_KEYWORDS = [keyword.strip() for keyword in content.split(',')]
except FileNotFoundError:
    KOREAN_KEYWORDS = ["로맨스", "판타지", "액션", "무협", "스릴러"]

ENGLISH_KEYWORDS = ["Romance Fantasy", "Modern Romance", "Traditional Fantasy", "Modern Fantasy", "Martial Arts", "Mystery", "Thriller", "Apocalypse", "Slice of Life", "RomCom", "Revenge", "Coming-of-age", "Healing", "Angst", "Comedy", "Calm", "Contract Dating", "Arranged Marriage", "First Love", "Mutual Salvation", "Misunderstanding", "Reverse Harem", "Harem", "Status Difference", "Capable FL", "Girl Crush FL", "Regression FL", "Reincarnation FL", "Possession FL", "Sweet ML", "Tsundere ML", "Cold Handsome ML", "Obsessive ML", "Crazy Obsessive ML", "Regretful ML", "Pure Love ML", "Straightforward ML", "Sly ML", "Scarred FL", "Mastermind ML", "Genius ML", "Satisfying Plot", "Brain Battle", "Psychological", "Plot Twist", "Solid World-building", "Foreshadowing Retrieval", "Solid Emotional Line", "Dark Fantasy", "Noir"]

def load_model():
    """
    모델을 로드하여 반환합니다. (양자화 완전히 제거)
    """
    # GPU 환경 확인 (0: 단일 GPU, -1: CPU)
    device = 0 if torch.cuda.is_available() else -1

    # 1. 번역 모델 로드
    # 🔥 에러 원인이었던 safetensors 옵션을 제거하고 PyTorch(pt) 사용을 명시합니다.
    translator = pipeline(
        "translation",
        model="Helsinki-NLP/opus-mt-ko-en",
        framework="pt",
        device=device
    )

    # 2. 제로샷 태깅 모델 로드 (양자화 제거)
    # 🔥 device_map="auto" 대신 명확하게 단일 GPU(device=0)를 지정합니다.
    tagger = pipeline(
        "zero-shot-classification",
        model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
        device=device
    )

    return {"translator": translator, "tagger": tagger}

def predict(model: dict, input_data: dict) -> dict:
    """
    입력을 받아 1~6 단계 로직을 수행하고 추론 결과를 반환합니다.
    """
    translator = model["translator"]
    tagger = model["tagger"]

    text = input_data["text"]
    title = input_data["title"]
    username = input_data["username"]

    # --- [로직 1] 파일명 생성 및 데이터 폴더 저장 ---
    os.makedirs(f"data/{username}", exist_ok=True)
    base_filename = datetime.now().strftime("%Y%m%d%H%M%S") + str(random.randint(100, 999))
    txt_path = f"data/{username}/{base_filename}.txt"

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(text)

    # --- [로직 1, 2] 문단/문장 분리 및 정제 ---
    paragraphs = text.split('\n')
    parsed_paragraphs = []

    for p in paragraphs:
        if not p.strip(): continue
        # 마침표(.) 단위로 문장 분리
        sentences = [s.strip() + '.' for s in p.split('.') if s.strip()]
        para_sents = []
        for s in sentences:
            # 불필요한 특수 문자 제거
            clean_s = re.sub(r'[^a-zA-Z0-9가-힣\s\'".,!?]', '', s).strip()
            if not clean_s: continue

            # 문장 타입 판별 (0=일반, 1=대사, 2=생각)
            s_type = 0
            if clean_s.startswith('"') and clean_s.endswith('"'):
                s_type = 1
            elif clean_s.startswith("'") and clean_s.endswith("'"):
                s_type = 2
            para_sents.append([s_type, clean_s])

        if para_sents:
            parsed_paragraphs.append(para_sents)

    # --- [로직 3, 4] 문단 단위 번역 & 제로샷 태깅 ---
    quantized_result = []
    all_orig_keywords = []
    all_trans_keywords = []

    for para in parsed_paragraphs:
        para_translated = []
        para_orig_text = ""
        para_trans_text = ""

        # 문단 내 문장별 번역 처리
        for s_type, orig_sent in para:
            trans_sent = translator(orig_sent)['translation_text']
            para_translated.append([s_type, orig_sent, trans_sent])
            para_orig_text += orig_sent + " "
            para_trans_text += trans_sent + " "

        # 문단 단위로 가장 확률 높은 키워드(상위 3개) 추출
        orig_tags = tagger(para_orig_text, KOREAN_KEYWORDS, multi_label=True)
        trans_tags = tagger(para_trans_text, ENGLISH_KEYWORDS, multi_label=True)

        para_orig_keys = orig_tags['labels'][:3]
        para_trans_keys = trans_tags['labels'][:3]

        all_orig_keywords.extend(para_orig_keys)
        all_trans_keywords.extend(para_trans_keys)

        # quantized_result 구조화
        quantized_result.append([para_translated, [para_orig_keys, para_trans_keys]])

    # --- [로직 5] 전체 키워드 상위 5개 추출 및 딕셔너리 구성 ---
    top_orig = [k for k, v in Counter(all_orig_keywords).most_common(5)]
    top_trans = [k for k, v in Counter(all_trans_keywords).most_common(5)]

    final_dict = {
        "title": title,
        "origin_txt": txt_path,
        "keyword": [top_orig, top_trans],
        "data": quantized_result
    }

    # --- [로직 6] JSON 형태로 동일한 폴더, 이름으로 저장 ---
    json_path = f"data/{username}/{base_filename}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(final_dict, f, ensure_ascii=False, indent=2)

    return final_dict

Overwriting app/model_service.py


#### ( try 4 ) 이제 제발 되자..!

In [6]:
%%writefile app/model_service.py
"""
모델 직접 로드 및 추론 함수 (TensorFlow 폴백 원천 차단 버전)
"""
import os
import re
import json
import random
import torch
from datetime import datetime
from collections import Counter
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForSequenceClassification

try:
    with open('data/keyword.txt', 'r', encoding='utf-8') as file:
        content = file.read()
        KOREAN_KEYWORDS = [keyword.strip() for keyword in content.split(',')]
except FileNotFoundError:
    KOREAN_KEYWORDS = ["로맨스", "판타지", "액션", "무협", "스릴러"]

ENGLISH_KEYWORDS = ["Romance Fantasy", "Modern Romance", "Traditional Fantasy", "Modern Fantasy", "Martial Arts", "Mystery", "Thriller", "Apocalypse", "Slice of Life", "RomCom", "Revenge", "Coming-of-age", "Healing", "Angst", "Comedy", "Calm", "Contract Dating", "Arranged Marriage", "First Love", "Mutual Salvation", "Misunderstanding", "Reverse Harem", "Harem", "Status Difference", "Capable FL", "Girl Crush FL", "Regression FL", "Reincarnation FL", "Possession FL", "Sweet ML", "Tsundere ML", "Cold Handsome ML", "Obsessive ML", "Crazy Obsessive ML", "Regretful ML", "Pure Love ML", "Straightforward ML", "Sly ML", "Scarred FL", "Mastermind ML", "Genius ML", "Satisfying Plot", "Brain Battle", "Psychological", "Plot Twist", "Solid World-building", "Foreshadowing Retrieval", "Solid Emotional Line", "Dark Fantasy", "Noir"]

def load_model():
    """
    AutoModel로 직접 PyTorch 모델을 로드하여 파이프라인에 주입합니다.
    """
    # 0번 GPU 사용, 없으면 CPU(-1)
    device = 0 if torch.cuda.is_available() else -1

    # ==========================================
    # 1. 번역 모델 (강제 PyTorch 로드)
    # ==========================================
    trans_model_id = "Helsinki-NLP/opus-mt-ko-en"
    trans_tokenizer = AutoTokenizer.from_pretrained(trans_model_id)
    trans_model = AutoModelForSeq2SeqLM.from_pretrained(trans_model_id)

    translator = pipeline(
        "translation",
        model=trans_model,
        tokenizer=trans_tokenizer,
        device=device
    )

    # ==========================================
    # 2. 제로샷 태깅 모델 (강제 PyTorch 로드)
    # ==========================================
    tagger_model_id = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
    tagger_tokenizer = AutoTokenizer.from_pretrained(tagger_model_id)
    tagger_model = AutoModelForSequenceClassification.from_pretrained(tagger_model_id)

    tagger = pipeline(
        "zero-shot-classification",
        model=tagger_model,
        tokenizer=tagger_tokenizer,
        device=device
    )

    return {"translator": translator, "tagger": tagger}

def predict(model: dict, input_data: dict) -> dict:
    """
    입력을 받아 로직을 수행하고 추론 결과를 반환합니다.
    """
    translator = model["translator"]
    tagger = model["tagger"]

    text = input_data["text"]
    title = input_data["title"]
    username = input_data["username"]

    os.makedirs(f"data/{username}", exist_ok=True)
    base_filename = datetime.now().strftime("%Y%m%d%H%M%S") + str(random.randint(100, 999))
    txt_path = f"data/{username}/{base_filename}.txt"

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(text)

    paragraphs = text.split('\n')
    parsed_paragraphs = []

    for p in paragraphs:
        if not p.strip(): continue
        sentences = [s.strip() + '.' for s in p.split('.') if s.strip()]
        para_sents = []
        for s in sentences:
            clean_s = re.sub(r'[^a-zA-Z0-9가-힣\s\'".,!?]', '', s).strip()
            if not clean_s: continue

            s_type = 0
            if clean_s.startswith('"') and clean_s.endswith('"'):
                s_type = 1
            elif clean_s.startswith("'") and clean_s.endswith("'"):
                s_type = 2
            para_sents.append([s_type, clean_s])

        if para_sents:
            parsed_paragraphs.append(para_sents)

    quantized_result = []
    all_orig_keywords = []
    all_trans_keywords = []

    for para in parsed_paragraphs:
        para_translated = []
        para_orig_text = ""
        para_trans_text = ""

        for s_type, orig_sent in para:
            trans_sent = translator(orig_sent)['translation_text']
            para_translated.append([s_type, orig_sent, trans_sent])
            para_orig_text += orig_sent + " "
            para_trans_text += trans_sent + " "

        orig_tags = tagger(para_orig_text, KOREAN_KEYWORDS, multi_label=True)
        trans_tags = tagger(para_trans_text, ENGLISH_KEYWORDS, multi_label=True)

        para_orig_keys = orig_tags['labels'][:3]
        para_trans_keys = trans_tags['labels'][:3]

        all_orig_keywords.extend(para_orig_keys)
        all_trans_keywords.extend(para_trans_keys)

        quantized_result.append([para_translated, [para_orig_keys, para_trans_keys]])

    top_orig = [k for k, v in Counter(all_orig_keywords).most_common(5)]
    top_trans = [k for k, v in Counter(all_trans_keywords).most_common(5)]

    final_dict = {
        "title": title,
        "origin_txt": txt_path,
        "keyword": [top_orig, top_trans],
        "data": quantized_result
    }

    json_path = f"data/{username}/{base_filename}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(final_dict, f, ensure_ascii=False, indent=2)

    return final_dict

Overwriting app/model_service.py


### 3-5.📝main.py — 직접 작성

In [24]:
%%writefile app/main.py
"""
FastAPI 서버 정의
"""
import asyncio
import traceback
from concurrent.futures import ThreadPoolExecutor
from fastapi import FastAPI, Depends, HTTPException

# 이전에 작성해둔 인증 모듈과 스키마 파일 로드
from app.auth import verify_api_key
from app.schemas import NarrativeRequest, NarrativeResponse
from app.model_service import load_model, predict

# 1. FastAPI 앱 생성
app = FastAPI(
    title="Localization Pipeline API",
    description="다국어 번역과 핵심 키워드(태그) 추출을 동시에 수행하는 지능형 파이프라인 API",
    version="1.0.0"
)

# 블로킹 방지를 위한 추론 전용 스레드풀 (Colab L4 환경 고려)
inference_executor = ThreadPoolExecutor(max_workers=2, thread_name_prefix="inference")

# 전역 모델 변수
pipeline_models = None

# 2. startup 이벤트에서 모델 로드
@app.on_event("startup")
async def startup_event():
    global pipeline_models
    try:
        print("모델 다운로드 중...")
        pipeline_models = load_model()
        print("✅ 모델 로드 완료")
    except Exception as e:
        print(f"❌ 모델 로드 실패: {e}")
        print("--- 상세 에러 트레이스 ---")
        traceback.print_exc() # 상세 에러 스택 로깅
        print("----------------------")
        pipeline_models = None

# 3. GET /health 엔드포인트
@app.get("/health", tags=["System"])
async def health_check():
    """서버 상태와 모델 로드 여부를 확인합니다."""
    return {
        "status": "healthy" if pipeline_models is not None else "loading",
        "model_loaded": pipeline_models is not None
    }

# 4. POST /predict 엔드포인트
@app.post("/predict", response_model=NarrativeResponse, tags=["Inference"])
async def predict_endpoint(
    request: NarrativeRequest,            # Pydantic 스키마로 5000자 길이 등 입력 검증
    user: str = Depends(verify_api_key)   # X-API-Key 헤더 인증 적용 및 username 획득
):
    """최대 5000자의 서사 텍스트를 분석하여 번역 및 키워드를 추출합니다."""

    if pipeline_models is None:
        raise HTTPException(
            status_code=503,
            detail="모델이 아직 로드되지 않았습니다. 서버 상태를 확인해주세요."
        )

    # 추론 서비스로 넘길 입력 데이터 구성
    input_data = {
        "title": request.title,
        "text": request.text,
        "username": user
    }

    try:
        # run_in_executor로 비동기 추론 (이벤트 루프 블로킹 방지)
        loop = asyncio.get_event_loop()
        result_dict = await loop.run_in_executor(
            inference_executor,
            predict,
            pipeline_models,
            input_data
        )

        # model_service.py의 반환 딕셔너리를 NarrativeResponse 스키마 구조에 맞게 매핑
        formatted_data = []
        for para_info in result_dict["data"]:
            sentences_list = para_info[0] # [[문장타입, 원문문장, 번역문장], ...]
            keywords_list = para_info[1]  # [원문키워드, 번역문장키워드]

            formatted_sentences = [
                {
                    "sentence_type": s_type,
                    "original_text": orig,
                    "translated_text": trans
                }
                for s_type, orig, trans in sentences_list
            ]

            formatted_data.append({
                "sentences": formatted_sentences,
                "keywords": keywords_list
            })

        # 최종 응답 객체 반환
        return NarrativeResponse(
            title=result_dict["title"],
            origin_txt=result_dict["origin_txt"],
            keyword=result_dict["keyword"],
            data=formatted_data
        )

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"파이프라인 처리 중 에러가 발생했습니다: {str(e)}")


Overwriting app/main.py


### 3-6.📝frontend/app.py — 직접 작성

In [29]:
%%writefile frontend/app.py
"""
Streamlit 프론트엔드
"""
import streamlit as st
import requests

# [API 엔드포인트 설정]
API_URL = "http://localhost:8000/predict"

# [세션 상태(Session State) 초기화]
if "temp_text" not in st.session_state:
    st.session_state["temp_text"] = ""
if "current_input" not in st.session_state:
    st.session_state["current_input"] = ""
if "api_result" not in st.session_state:
    st.session_state["api_result"] = None
if "history_works" not in st.session_state:
    st.session_state["history_works"] = []

# ---------------------------------------------------------
# 1. 페이지 및 사이드바 설정 [1, 3]
# ---------------------------------------------------------
st.set_page_config(page_title="지능형 현지화 파이프라인", layout="wide")

with st.sidebar:
    st.header("⚙️ 설정")
    # 사이드바에 API Key 입력 [3]
    api_key = st.text_input("API Key 입력", type="password", help="발급받은 API 키를 입력하세요.")
    st.divider()
    # 그동안 작성한 작품 리스트 선택
    st.selectbox("내가 작성한 작품 리스트", ["선택하세요..."] + st.session_state["history_works"])

# ---------------------------------------------------------
# 2. 메인 화면 - 서비스 소개 및 입력부 [1, 4]
# ---------------------------------------------------------
# st.title()로 제목
st.title("지능형 현지화 파이프라인")
st.markdown("최대 5000자의 서사 텍스트를 입력받아 **다국어 번역**과 **핵심 키워드 추출**을 수행하는 프로토타입입니다.")

# 작품 제목 입력창
work_title = st.text_input("작품 제목", placeholder="작품의 제목을 입력해주세요.")

# 분석 실행을 위한 공통 함수 (버튼 클릭 시 requests.post()로 API 호출) [1, 2]
def analyze_text(title, text):
    if not title or not text:
        st.warning("작품 제목과 텍스트를 모두 입력해주세요.")
        return
    if not api_key:
        st.warning("우측 사이드바에 API Key를 입력해주세요.")
        return

    with st.spinner("번역 및 키워드 추출을 진행 중입니다..."):
        headers = {"X-API-Key": api_key}
        payload = {"title": title, "text": text}
        try:
            response = requests.post(API_URL, json=payload, headers=headers)
            response.raise_for_status()

            # 결과 저장 및 히스토리 업데이트
            st.session_state["api_result"] = response.json()
            if title not in st.session_state["history_works"]:
                st.session_state["history_works"].append(title)

            st.success("분석이 완료되었습니다!")
        except requests.exceptions.HTTPError as e:
            st.error(f"API 호출 실패 (HTTP {e.response.status_code}): {e.response.text}")
        except Exception as e:
            st.error(f"서버에 연결할 수 없습니다: {e}")

# 탭 구성: 파일로 불러오기 / 작품 입력하기
tab1, tab2 = st.tabs(["📁 파일로 불러오기", "✍️ 작품 입력하기"])

with tab1:
    # 본인의 모델에 맞는 입력 위젯 (file_uploader)
    uploaded_file = st.file_uploader("분석할 텍스트 파일(.txt)을 업로드하세요", type=["txt"])
    file_text = ""
    if uploaded_file:
        file_text = uploaded_file.read().decode("utf-8")
        st.text_area("파일 내용 미리보기", file_text, height=150, disabled=True)

    if st.button("확인 (파일 분석)"):
        analyze_text(work_title, file_text)

with tab2:
    # 본인의 모델에 맞는 입력 위젯 (text_input/text_area)
    current_input = st.text_area(
        "서사 텍스트 입력 (최대 5000자)",
        value=st.session_state["current_input"],
        height=300,
        max_chars=5000
    )
    # 글자수 세기
    st.caption(f"현재 글자수: {len(current_input)} / 5000자")

    col1, col2, col3 = st.columns(3)
    with col1:
        if st.button("💾 임시저장"):
            st.session_state["temp_text"] = current_input
            st.success("상태(Status)에 임시저장되었습니다.")
    with col2:
        if st.button("📂 임시저장 파일 불러오기"):
            st.session_state["current_input"] = st.session_state["temp_text"]
            st.rerun()  # 화면을 새로고침하여 불러온 텍스트 반영 [5]
    with col3:
        if st.button("확인 (텍스트 분석)"):
            st.session_state["current_input"] = current_input
            analyze_text(work_title, current_input)

# ---------------------------------------------------------
# 3. 작품 출력 화면 (결과 표시) [4, 6]
# ---------------------------------------------------------
if st.session_state["api_result"]:
    st.divider()
    result = st.session_state["api_result"]

    # 작품 제목
    st.header(f"📖 {result['title']}")

    # 제어 옵션 레이아웃
    ctrl_col1, ctrl_col2 = st.columns(2)
    with ctrl_col1:
        # 출력 내용 선택
        view_option = st.radio(
            "출력 내용 선택",
            ["원문보기", "번역본보기", "원문+번역 대조보기"],
            horizontal=True
        )
    with ctrl_col2:
        # 출력 형태 선택 (p태그 display 제어)
        format_option = st.radio(
            "출력 형태 선택",
            ["Block (단락별 줄바꿈)", "Inline (자연스럽게 이어쓰기)"],
            horizontal=True
        )

    # 작품 키워드 출력 (출력 형태에 따라 분기)
    st.subheader("🔑 작품 핵심 키워드")
    top_orig_kw = result["keyword"] if len(result["keyword"]) > 0 else []
    top_trans_kw = result["keyword"][7] if len(result["keyword"]) > 1 else []

    if view_option == "원문보기":
        st.write(f"**원문 키워드:** {', '.join(top_orig_kw)}")
    elif view_option == "번역본보기":
        st.write(f"**번역본 키워드:** {', '.join(top_trans_kw)}")
    else:
        st.write(f"**원문 키워드:** {', '.join(top_orig_kw)}")
        st.write(f"**번역본 키워드:** {', '.join(top_trans_kw)}")

    st.markdown("---")

    # ---------------------------------------------------------
    # 4. 동적 CSS 및 HTML 렌더링
    # ---------------------------------------------------------
    # 사용자가 Inline을 선택하면 normal 타입 p태그는 inline, Block이면 block.
    # 대사(dialogue)와 생각(thought)은 무조건 display: block.
    wrapper_class = "block-mode" if format_option == "Block (단락별 줄바꿈)" else "inline-mode"

    custom_css = f"""
    <style>
    /* 기본 컨테이너 및 단락(<section>) 스타일 */
    .render-container {{ font-size: 16px; line-height: 1.6; }}
    section.paragraph {{ margin-bottom: 20px; padding: 15px; background-color: #f8f9fa; border-radius: 8px; }}

    /* 일반 문장(normal) 동적 디스플레이 */
    .block-mode p.normal {{ display: block; margin-bottom: 10px; }}
    .inline-mode p.normal {{ display: inline; margin-right: 5px; }}

    /* 대사(dialogue)와 생각(thought)은 무조건 block, 따옴표 생성 */
    p.dialogue, p.thought {{
        display: block !important;
        margin: 10px 0;
        padding-left: 10px;
    }}
    p.dialogue {{ font-weight: 600; color: #1565C0; }}
    p.dialogue::before {{ content: '"'; }}
    p.dialogue::after {{ content: '"'; }}

    p.thought {{ font-style: italic; color: #6A1B9A; }}
    p.thought::before {{ content: "'"; }}
    p.thought::after {{ content: "'"; }}

    /* 번역본 대조보기 시 번역 텍스트 색상 처리 */
    .translated-text {{ color: #757575; font-size: 0.95em; display: block; margin-top: 2px; }}
    </style>
    """

    # HTML 조립 (문단은 <section>, 문장은 <p> 태그)
    html_out = f'{custom_css}<div class="render-container {wrapper_class}">'

    for para in result["data"]:
        html_out += '<section class="paragraph">'

        for sent in para["sentences"]:
            s_type = sent["sentence_type"]
            orig_text = sent["original_text"]
            trans_text = sent.get("translated_text", "")

            # 클래스 맵핑
            if s_type == 1:
                p_class = "dialogue"
            elif s_type == 2:
                p_class = "thought"
            else:
                p_class = "normal"

            # 표시 내용 분기
            if view_option == "원문보기":
                disp_text = orig_text
            elif view_option == "번역본보기":
                disp_text = trans_text
            else:
                # 대조보기: 원문 아래에 번역문을 별도 span으로 배치
                disp_text = f"{orig_text}<span class='translated-text'>{trans_text}</span>"

            html_out += f'<p class="{p_class}">{disp_text}</p>'

        html_out += '</section>'

    html_out += '</div>'

    # 렌더링
    st.markdown(html_out, unsafe_allow_html=True)

Writing frontend/app.py


## 4. 작업 시간

### 4-2. 서버실행

#### ( try 1 )

In [30]:
# ⚠️ 커널 재시작 후 실행하세요
import nest_asyncio, uvicorn, threading, time
nest_asyncio.apply()

def run_server():
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ 서버 시작됨 — http://localhost:8000/docs")

Exception in thread Thread-5 (run_server):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_35076/1634802692.py", line 7, in run_server
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/main.py", line 577, in run
    server.run()
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/server.py", line 65, in run
    return asyncio.run(self.serve(sockets=sockets))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 26, in run
    loop = asyncio.get_event_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/nest_asyncio.py", line 40, in _get_event_loop
    loop = events.get_event_loop_policy().get_event_loop()
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

✅ 서버 시작됨 — http://localhost:8000/docs


#### ( try 2 ) 코랩용으로 수정

##### ☢️ 모델 로드 문제
- transformers 라이브러리가 보안을 위해 구버전 PyTorch에서의 모델 로딩(Pickle 파일)을 강제로 차단
- 파이토치(PyTorch) 로드에 실패한 후 텐서플로우(TensorFlow)로 로드를 재시도했으나, 텐서플로우는 device_map이라는 인자를 지원하지 않아(Keyword argument not understood: device_map) 최종적으로 모든 방식에서 실패

##### 💊해결방법
- PyTorch 버전을 최신(2.6 이상)으로 업데이트하여 해결

In [41]:
!pip install --upgrade torch torchvision torchaudio transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 156.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.

##### 서버실행

In [ ]:
!fuser -k 8000/tcp

In [8]:
import nest_asyncio, uvicorn, threading, time, asyncio
nest_asyncio.apply()

def run_server():
    # 1. 새 스레드를 위한 이벤트 루프를 명시적으로 생성
    loop = asyncio.new_event_loop()
    # 2. 생성한 루프를 현재 스레드의 기본 루프로 설정
    asyncio.set_event_loop(loop)
    # 3. Uvicorn 서버 실행 (loop="asyncio" 추가하여 uvloop 충돌 방지)
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000, loop="asyncio")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(60) # 모델 로드 시간을 고려해 10초 대기를 권장합니다.
print("✅ 서버 시작됨 — http://localhost:8000/docs")

INFO:     Started server process [9705]
INFO:     Waiting for application startup.


모델 다운로드 중...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Pr

✅ 모델 로드 완료
✅ 서버 시작됨 — http://localhost:8000/docs


In [9]:
from google.colab import output
output.serve_kernel_port_as_iframe(8000, path='/docs')

<IPython.core.display.Javascript object>

In [10]:
# 주의: app/frontend.py 부분은 작성하신 Streamlit 파일의 실제 경로/이름으로 맞춰주세요!
!nohup streamlit run app/frontend.py &> frontend_log.txt &

In [12]:
from google.colab import output

# iframe 대신 새 탭(Window)으로 열어 진짜 웹사이트처럼 넓게 씁니다.
# output.serve_kernel_port_as_window(8501)

# (만약 기존처럼 Colab 내부에서 작게 보고 싶으시다면 아래 코드 사용)
output.serve_kernel_port_as_iframe(8501, height=800)

<IPython.core.display.Javascript object>